# Import and Setup Wavlm

In [1]:
import string
import IPython
from IPython.display import Audio
import torch
import os

import torchaudio

from TTS.tts.utils.synthesis import synthesis
from TTS.utils.audio import AudioProcessor
from TTS.tts.models import setup_model
from TTS.config import load_config
from TTS.tts.models.vits import *
from TTS.tts.utils.speakers import SpeakerManager
from TTS.utils.vad import get_vad_model_and_utils, remove_silence, resample_wav, read_audio

from pydub import AudioSegment

from transformers import Wav2Vec2FeatureExtractor, WavLMForXVector

OUT_PATH = 'output'

In [2]:
def extract_wavlm_embedding(wav_file: str, device='cuda', use_cuda=True) -> list:
    """Compute a embedding from a given audio file using WavLM.

    Args:
        model: WavLM model
        feature_extractor: WavLM feature extractor
        wav_file (Union[str, List[str]]): Target file path or list of paths
        device (str): Computing device

    Returns:
        list: Computed embedding
    """
    
    device = 'cuda' if use_cuda and torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
    
    # Load model and feature extractor
    model = WavLMForXVector.from_pretrained("microsoft/wavlm-base-plus-sv").to(device)
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base-plus-sv")
    
    model.eval()
    
    def _compute(wav_file: str):
        # Load and resample audio if necessary
        waveform, sample_rate = torchaudio.load(wav_file)
        if sample_rate != 16000:
            resampler = torchaudio.transforms.Resample(sample_rate, 16000)
            waveform = resampler(waveform)
        
        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        # Convert to numpy array for feature extractor
        audio_array = waveform.squeeze().numpy()
        
        # Process through WavLM
        inputs = feature_extractor(audio_array, sampling_rate=16000, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            
        # Normalize embeddings
        embedding = torch.nn.functional.normalize(outputs.embeddings, dim=-1)
        return embedding

    if isinstance(wav_file, list):
        # Compute the mean embedding for multiple files
        embeddings = None
        for wf in wav_file:
            embedding = _compute(wf)
            if embeddings is None:
                embeddings = embedding
            else:
                embeddings += embedding
        return (embeddings / len(wav_file))[0].cpu().tolist()
    
    embedding = _compute(wav_file)
    return embedding[0].cpu().tolist()


# Test Argument

## Test 1

In [3]:
# OUT_PATH = 'output'
# REFERENCE_FILENAMES = ['cv049_028_mic1.flac',
#                        'cv017_012_mic1.flac']

# texts = ["กล้วยตานีปลายหวีเหี่ยวหิ้วหวีไปหิ้วหวีมา",
#          "ใครใครขายไข่ไก่ใกล้ค่าย",
#          "ยายกินลำไย น้ำลายยายไหลย้อย",
#          "เช้าฟาดผัดฟัก เย็นฟาดฟักผัด"]

# # model vars 

# MODELS = [
#     {'model': ['best_model_160217', 'checkpoint_453866'], 'model_path': './model/tsync2_cv_1'},
#     {'model': ['best_model_62921', 'best_model_277421', 'checkpoint_240000', 'checkpoint_400000'], 'model_path': './model/tsync2_cv_2'},
# ]

# test = "test1"

# isRestructure = False

## Test 2

In [4]:
# OUT_PATH = 'output'
# REFERENCE_FILENAMES = ['cv049_028_mic1.flac',
#                        'cv017_012_mic1.flac',
#                        'Jinny-04-en.m4a']

# texts = ["เช้าฟาดผัดฟัก เย็นฟาดฟักผัด",
#          "ยายกินลำไย น้ำลายยายไหลย้อย",
#          "ทรวดทรงซาบทรามทราย",
#          "ตามรายงานข่าววันนี้ มีการประชุมสำคัญเกี่ยวกับอนาคตของเทคโนโลยีปัญญาประดิษฐ์ โดยผู้เชี่ยวชาญจากหลายประเทศเข้าร่วมอภิปรายเกี่ยวกับแนวโน้มและผลกระทบที่อาจเกิดขึ้น",
#          "เขายืนอยู่ใต้แสงจันทร์ นึกถึงวันที่เคยมีเธออยู่ข้างข้าง ทุกความทรงจำยังชัดเจน ราวกับว่าเวลาหยุดเดินไปพร้อมกับหัวใจของเขา"]

# # model vars 

# MODELS = [
#           {'model':['best_model_160217'], 'model_path':'./model/tsync2_cv_1'},
#           {'model':['checkpoint_240000'], 'model_path':'./model/tsync2_cv_2'},
#           {'model':['best_model_76315',
#           'best_model_167165',
#           'checkpoint_120000',
#           'checkpoint_230000',
#           'checkpoint_400000',
#           'checkpoint_460000',
#           'checkpoint_480000',
#           'checkpoint_495611'], 'model_path':'./model/tsync2_cv_wavlm'},
#          ]

# test = "test2"

## Test 3

In [5]:
# OUT_PATH = 'output'
# REFERENCE_FILENAMES = ['cv049_028_mic1.flac',
#                        'cv017_012_mic1.flac',
#                        'Jinny-04-en.m4a']

# texts = ["เช้าฟาดผัดฟัก เย็นฟาดฟักผัด",
#          "ยายกินลำไย น้ำลายยายไหลย้อย",
#          "ทรวดทรงซาบทรามทราย",
#          "ตามรายงานข่าววันนี้ มีการประชุมสำคัญเกี่ยวกับอนาคตของเทคโนโลยีปัญญาประดิษฐ์ โดยผู้เชี่ยวชาญจากหลายประเทศเข้าร่วมอภิปรายเกี่ยวกับแนวโน้มและผลกระทบที่อาจเกิดขึ้น",
#          "เขายืนอยู่ใต้แสงจันทร์ นึกถึงวันที่เคยมีเธออยู่ข้างข้าง ทุกความทรงจำยังชัดเจน ราวกับว่าเวลาหยุดเดินไปพร้อมกับหัวใจของเขา",
#          "กล้วยตานีปลายหวีเหี่ยวหิ้วหวีไปหิ้วหวีมา"]

# # model vars 

# MODELS = [
#     {'model': ['best_model_160217'], 'model_path': './model/tsync2_cv_1'},
#     {'model': ['checkpoint_240000'], 'model_path': './model/tsync2_cv_2'},
# ]

# test = "test3"

## Test 4

In [6]:
# REFERENCE_FILENAMES = ['cv049_028_mic1.flac',
#                        'cv017_012_mic1.flac',
#                        'Jinny-04-en.m4a',
#                        'Ming-12-th.wav',
#                        'ajred_cut.wav',
#                        'ekapol_cut.wav']

# texts = ["เช้าฟาดผัดฟัก เย็นฟาดฟักผัด",
#          "ยายกินลำไย น้ำลายยายไหลย้อย",
#          "ทรวดทรงซาบทรามทราย",
#          "ตามรายงานข่าววันนี้ มีการประชุมสำคัญเกี่ยวกับอนาคตของเทคโนโลยีปัญญาประดิษฐ์ โดยผู้เชี่ยวชาญจากหลายประเทศเข้าร่วมอภิปรายเกี่ยวกับแนวโน้มและผลกระทบที่อาจเกิดขึ้น",
#          "เขายืนอยู่ใต้แสงจันทร์ นึกถึงวันที่เคยมีเธออยู่ข้างข้าง ทุกความทรงจำยังชัดเจน ราวกับว่าเวลาหยุดเดินไปพร้อมกับหัวใจของเขา",
#          "กล้วยตานีปลายหวีเหี่ยวหิ้วหวีไปหิ้วหวีมา"]

# # model vars 

# MODELS = [
#     {'model': ['best_model_160217', 'checkpoint_453866'], 'model_path': './model/tsync2_cv_1'},
#     {'model': ['best_model_62921', 'best_model_277421', 'checkpoint_240000', 'checkpoint_400000'], 'model_path': './model/tsync2_cv_2'},
# ]

# test = "test4"
# isRestructure = False

## Test 5

In [7]:
# REFERENCE_FILENAMES = ['cv131_046_mic1.flac',
#                        'cv058_040_mic1.flac',
#                        'kirito.wav',
#                        'azato.wav']

# texts = ["เตือนไม่ควรรับสายแล้วพูดคนเดียวนานนาน เพราะมิจฉาชีพบันทึกเสียงแล้วใช้ เอไอ ปลอมเสียงแบบวอยซ์โคลน หลอกญาติพี่น้องคนสนิทได้",
#          "นายกรัฐมนตรี ให้สัมภาษณ์ถึงกรณีที่มีแก๊งคอลเซนเตอร์แอบอ้างเป็นเสียงของผู้นำต่างประเทศส่งข้อความเสียงมาว่า เป็นประเทศในอาเซียนที่ยังไม่ได้รับเงินบริจาค",
#          "ลิงใหญ่ ยกลำไยเล็ก ลิงเล็ก ยกลำไยใหญ่",
#          "กินมันติดเหงือก กินเผือกติดฟัน",
#          "หมู หมึก กุ้ง หุง อุ่น หุง ต้ม ตุ๋น อุ่น นึ่ง",
#          "ก็พบกันอีกเช่นเคยนะครับ สำหรับท่านที่เดินผ่านไปผ่านมานะครับ วันนี้ เฉาก๊วยชากังราวของเรานะครับ ก็ได้มาบริการท่านพ่อแม่พี่น้องอีกแล้วครับ"]

# # model vars 

# MODELS = [
#     {'model': ['checkpoint_453866'], 'model_path': './model/tsync2_cv_1'},
#     {'model': ['checkpoint_220000', 'checkpoint_455000', 'checkpoint_555000', 'checkpoint_615000'], 'model_path': './model/tsync2_cv_1_vs'},
# ]

# test = "test5"
# isRestructure = True

## Test 6

In [8]:
# REFERENCE_FILENAMES = ['cv131_046_mic1.flac',
#                        'cv058_040_mic1.flac',
#                        'kirito.wav',
#                        'azato.wav']

# texts = ["เตือนไม่ควรรับสายแล้วพูดคนเดียวนานนาน เพราะมิจฉาชีพบันทึกเสียงแล้วใช้ เอไอ ปลอมเสียงแบบวอยซ์โคลน หลอกญาติพี่น้องคนสนิทได้",
#          "นายกรัฐมนตรี ให้สัมภาษณ์ถึงกรณีที่มีแก๊งคอลเซนเตอร์แอบอ้างเป็นเสียงของผู้นำต่างประเทศส่งข้อความเสียงมาว่า เป็นประเทศในอาเซียนที่ยังไม่ได้รับเงินบริจาค",
#          "ลิงใหญ่ ยกลำไยเล็ก ลิงเล็ก ยกลำไยใหญ่",
#          "กินมันติดเหงือก กินเผือกติดฟัน",
#          "หมู หมึก กุ้ง หุง อุ่น หุง ต้ม ตุ๋น อุ่น นึ่ง",
#          "ก็พบกันอีกเช่นเคยนะครับ สำหรับท่านที่เดินผ่านไปผ่านมานะครับ วันนี้ เฉาก๊วยชากังราวของเรานะครับ ก็ได้มาบริการท่านพ่อแม่พี่น้องอีกแล้วครับ"]

# # model vars 
# MODELS = [
#     {'model': ['checkpoint_453866'], 'model_path': './model/tsync2_cv_1'},
#     {'model': ['best_model_277421', 'checkpoint_495000', 'checkpoint_500000'], 'model_path': './model/tsync2_cv_no-my'},
#     {'model': ['best_model_223081', 'checkpoint_235000', 'checkpoint_510000', 'checkpoint_675000', 'checkpoint_725000', 'checkpoint_730000'], 'model_path': './model/tsync2_cv_rw'},
# ]

# test = "test6"
# isRestructure = True

## Test 7

In [9]:
# REFERENCE_FILENAMES = ['cv131_046_mic1.flac',
#                        'cv058_040_mic1.flac',
#                        'jeen.wav',
#                        'luna_3.wav',
#                        'gokuth.wav'
#                        ]

# texts = ["ไม่เพียงแต่จะเป็นเครื่องมือสำหรับการเรียนการสอน แต่ยังเป็นสัญลักษณ์ของความก้าวหน้าทางเทคโนโลยีเพื่อการพัฒนาอย่างยั่งยืน",
#          "โพสต์ดังกล่าว มีชาวเน็ตเข้ามาแสดงความคิดเห็นมากมาย จนเสียงแตกเป็นสองฝั่ง บางคนมองว่าผู้ใช้งานรายนี้กำลังแสดงให้เห็นว่าชอบลัดขั้นตอน แต่บางคนชื่นชมในความคิดสร้างสรรค์และมองว่ายุติธรรม",
#          "การจำกัดและกำจัดซากเศษวัสดุในพื้นที่ซึ่งสำนักงานเขตรับผิดชอบนั้น จำเป็นต้องรีบดำเนินการอย่างเร่งด่วน เพราะอาจก่อให้เกิดปัญหาการระบาดของเชื้อโรคซึ่งสัตว์พาหะนำโรค",
#          "ทรุดโทรมหมายนกอินทรี",
#          "เศรษฐีสร้างสร้อยคอระหว่างกินพุทราหน้ากองทรายใต้ต้นไทร"]

# # model vars 
# MODELS = [
#     {'model': ['checkpoint_453866'], 'model_path': './model/tsync2_cv_1'},
#     {'model': ['checkpoint_500000'], 'model_path': './model/tsync2_cv_no-my'},
#     {'model': ['cv_cont_vs_checkpoint_535000', 'cv_cont_vs_checkpoint_730000', 'cv_cont_vs_checkpoint_820000', 'cv_cont_vs_checkpoint_985000'], 'model_path': './model/tsync2_cv_cont_vs'},
# ]

# test = "test7"
# isRestructure = True

## Test 8

In [10]:
REFERENCE_FILENAMES = ['Tsync2_905_mic1.flac'
                       ]

texts = ["ใครใคร่ขายไข่ไก่ใกล้ค่าย",
         "ทรวดทรงทราบทรามทราย",
         "หมูหมึกกุ้ง หุงอุ่นตุ๋นต้มนึ่ง"]

# model vars 
MODELS = [
    {'model': ['best_model_28085', 'checkpoint_110000', 'checkpoint_285000',
               'checkpoint_410000', 'checkpoint_413001'], 'model_path': './model/base_th_en'},
    {'model': ['best_model_26881', 'best_model_34777', 'best_model_137329',
               'checkpoint_80000', 'checkpoint_135000'], 'model_path': './model/tsync2'},
]

test = "test8"
isRestructure = True

# Check if all files exist

In [11]:
# Check if all model files exist
for model in MODELS:
    for model_name in model['model']:
        model_path = os.path.join(model['model_path'], model_name + '.pth')
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Model file not found: {model_path}")

# Check if all reference files exist
for ref_file in REFERENCE_FILENAMES:
    ref_path = os.path.join("./reference_voice", ref_file)
    ref_wav_path = os.path.join("./reference_voice", "wav", ref_file.replace('.flac', '.wav'))
    if not os.path.exists(ref_path):
        raise FileNotFoundError(f"Reference file not found: {ref_path}")

# Process Reference voices

In [12]:
USE_CUDA = torch.cuda.is_available()
REFERENCE_PATHS = [os.path.join("./reference_voice", REFERENCE_FILENAME.strip().strip('./').strip('/')) for REFERENCE_FILENAME in REFERENCE_FILENAMES]
REFERENCE_WAV_PATHS = [os.path.join("./reference_voice", "wav", REFERENCE_FILENAME.strip().strip('./').strip('/').split(".")[0] + ".wav") for REFERENCE_FILENAME in REFERENCE_FILENAMES]
SPEAKER_FOLDERS = [REFERENCE_PATH.split("/")[-1].split(".")[0] for REFERENCE_PATH in REFERENCE_PATHS]

new_reference_paths = []
  
# Process reference voices
for REFERENCE_PATH, REFERENCE_WAV_PATH in zip(REFERENCE_PATHS, REFERENCE_WAV_PATHS):
    # Check if refernec voice is in wav format
    current_ref_extension = REFERENCE_PATH.split(".")[-1].lower()

    # Convert to wav if not or never converted
    if current_ref_extension != "wav":
      if not os.path.exists(REFERENCE_WAV_PATH):
        audio: AudioSegment = AudioSegment.from_file(REFERENCE_PATH, format=current_ref_extension)
        audio.export(REFERENCE_WAV_PATH, format="wav")
      REFERENCE_PATH = REFERENCE_WAV_PATH
      
    new_reference_paths.append(REFERENCE_PATH)
    
    # Resmapling reference voice if needed
    ref_wav, current_sr = read_audio(REFERENCE_PATH)
    if current_sr != 16000:
        print('Resampling reference audio...')
        resamapled_ref_wav = resample_wav(ref_wav, current_sr, 16000)
        torchaudio.save(REFERENCE_PATH, resamapled_ref_wav[None, :], 16000)
    else:
        print('This audio is already in the correct sample rate')
    
    # trim silence at the beginning and end of the audio
    model_and_utils = get_vad_model_and_utils(use_cuda=USE_CUDA, use_onnx=False)
    output_path, is_speech = remove_silence(
      model_and_utils,
      REFERENCE_PATH,
      REFERENCE_PATH,
      trim_just_beginning_and_end=True,
      use_cuda=USE_CUDA
    )

    # normalize the reference audio with rms to -27dB
    !ffmpeg-normalize $REFERENCE_PATH -nt rms -t=-27 -o $REFERENCE_PATH -ar 16000 -f
    
REFERENCE_PATHS = new_reference_paths

This audio is already in the correct sample rate


Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /Users/jackkahod/.cache/torch/hub/master.zip


# Inference

In [13]:
from tqdm import tqdm

for MODEL in MODELS:
  MODEL_PATHS = [os.path.join(MODEL['model_path'], m+'.pth') for m in MODEL['model']]
  BASE_MODEL_PATH = MODEL['model_path']
  MODEL_NAMES = MODEL['model']
  CONFIG_PATH = os.path.join(MODEL['model_path'], 'config.json')
  TTS_LANGUAGES = os.path.join(MODEL['model_path'], 'language_ids.json')

  # Synthesize voice
  for MODEL_PATH, model_name in tqdm(list(zip(MODEL_PATHS, MODEL_NAMES)), desc="Processing models"):
    # load the config
    C = load_config(CONFIG_PATH)

    # load the audio processor
    ap = AudioProcessor(**C.audio)

    # override config
    C["speakers_file"] = None
    C["d_vector_file"] = []
    C["language_ids_file"] = TTS_LANGUAGES

    C["model_args"]["speakers_file"] = None
    C["model_args"]["d_vector_file"] = []
    C["model_args"]["language_ids_file"] = TTS_LANGUAGES

    C.model_args['use_speaker_encoder_as_loss'] = False

    model = setup_model(C)
    cp = torch.load(MODEL_PATH, map_location=torch.device('cpu'))

    # remove speaker encoder
    model_weights = cp['model'].copy()
    for key in list(model_weights.keys()):
      if "speaker_encoder" in key:
        del model_weights[key]

    model.load_state_dict(model_weights)

    model.eval()

    if USE_CUDA:
      model = model.cuda()

    # Select language
    language_id = 0
    language_name_to_id = model.language_manager.name_to_id
    language_id_to_name = {v: k for k, v in language_name_to_id.items()}

    model.length_scale = 1.13  # scaler for the duration predictor. The larger it is, the slower the speech.
    model.inference_noise_scale = 0.2 # defines the noise variance applied to the random z vector at inference.
    model.inference_noise_scale_dp = 0.2 # defines the noise variance applied to the duration predictor z vector at inference.

    for REFERENCE_PATH, SPEAKER_FOLDER in tqdm(list(zip(REFERENCE_PATHS, SPEAKER_FOLDERS)), desc="Processing speakers"):
      try:
        SE_speaker_manager = SpeakerManager(encoder_model_path=C["model_args"]["speaker_encoder_model_path"], encoder_config_path=C["model_args"]["speaker_encoder_config_path"], use_cuda=USE_CUDA)
        reference_emb = SE_speaker_manager.compute_embedding_from_clip(REFERENCE_PATH)
      except:
        reference_emb = extract_wavlm_embedding(REFERENCE_PATH, 'cpu')
      
      for text in tqdm(texts, desc="Generating audio"):
        wav, alignment, _, _ = synthesis(
                            model = model,
                            text = text,
                            CONFIG = C,
                            use_cuda = USE_CUDA,
                            d_vector = reference_emb,
                            style_wav = None,
                            language_id = language_id,
                            use_griffin_lim = False,
                            do_trim_silence = False,
                        ).values()

        file_name = text.replace(" ", "_")
        file_name = model_name + '_' + file_name.translate(str.maketrans('', '', string.punctuation.replace('_', ''))) + '.wav'
        
        out_path = f"{OUT_PATH}/{test}/{SPEAKER_FOLDER}/{BASE_MODEL_PATH.split('/')[-1]}/{MODEL_PATH.split('/')[-1].split('.')[0]}/{file_name}"
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        ap.save_wav(wav, out_path)

Processing models:   0%|          | 0/5 [00:00<?, ?it/s]


Using model: ./model/base_th_en/best_model_28085.pth
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_m

 > Model fully restored. 
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:64
 | > log_func:np.log10
 | > min_level_db:-100
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:20
 | > fft_size:512
 | > power:1.5
 | > preemphasis:0.97
 | > griffin_lim_iters:60
 | > signal_norm:False
 | > symmetric_norm:False
 | > mel_fmin:0
 | > mel_fmax:8000.0
 | > pitch_fmin:1.0
 | > pitch_fmax:640.0
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:4.0
 | > clip_norm:False
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:True
 | > db_level:-27.0
 | > stats_path:None
 | > base:10
 | > hop_length:160
 | > win_length:400



Generating: ใครใคร่ขายไข่ไก่ใกล้ค่าย



Generating: ทรวดทรงทราบทรามทราย


...


Generating: ใครใคร่ขายไข่ไก่ใกล้ค่าย



Generating: ทรวดทรงทราบทรามทราย



Generating: หมูหมึกกุ้ง หุงอุ่นตุ๋นต้มนึ่ง



Processing models:  60%|██████    | 3/5 [00:21<00:13,  6.98s/it]


Using model: ./model/base_th_en/checkpoint_410000.pth
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:16000
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_

# Restructure

In [14]:
if isRestructure:
    import os
    import random
    import string
    import shutil
    import json

    # Model variables (already declare in above cell)
    # MODELS = [
    #     {'model': ['best_model_160217', 'checkpoint_453866'], 'model_path': './model/tsync2_cv_1'},
    #     {'model': ['best_model_62921', 'best_model_277421', 'checkpoint_240000', 'checkpoint_400000'], 'model_path': './model/tsync2_cv_2'},
    # ]
    # # Reference names
    # REFERENCE_FILENAMES = ['cv049_028_mic1.flac',
    #                        'cv017_012_mic1.flac',
    #                        'Jinny-04-en.m4a',
    #                        'Ming-12-th.wav']
    # test = "test4"

    all_model = [model['model'] for model in MODELS]
    all_model = [model_name for model in all_model for model_name in model]

    # Function to get shuffled letters
    def get_shuffled_alphabet():
        letters = list(string.ascii_lowercase[:len(all_model)])
        random.shuffle(letters)
        return letters

    MODELS = [ {'model':m['model'], 'model_path':m['model_path'].lstrip('./model/')} for m in MODELS]

    REFERENCE_FILENAMES = [i.split('.')[0] for i in REFERENCE_FILENAMES]

    script_mapping = {}

    # Get shuffled alphabet for model representation
    shuffled_letters = get_shuffled_alphabet()
    all_models_names = [model['model_path'] + '/' + model_name for model in MODELS for model_name in model['model']]
    model_letter_map = {model: shuffled_letters[i] for i, model in enumerate(all_models_names)}

    # Define base output path for inference files
    input_base_path = f"./output/{test}"
    output_base_path = f"./output/blindtest/{test}/"
    os.makedirs(output_base_path, exist_ok=True)

    # Initialize a mapping dictionary
    ref_mapping = {}

    ref_id = 0

    # Process each reference name and its corresponding model directories
    for ref_name in REFERENCE_FILENAMES:
        ref_id += 1
        ref_mapping[ref_id] = ref_name
        for model_name in all_models_names:
            model_letter = model_letter_map[model_name]

            # Construct the model directory path
            model_set_path = os.path.join(input_base_path, ref_name, model_name)

            if not os.path.exists(model_set_path):
                continue

            inference_files = os.listdir(model_set_path)
            script_list = []
            for script_num, file in enumerate(inference_files, start=1):
                if not file.endswith('.wav'):
                    continue
                
                if file.rstrip('.wav').lstrip(model_name + '_') not in script_mapping:
                    script_mapping[file.rstrip('.wav').lstrip(model_name + '_')] = script_num
                else:
                    script_num = script_mapping[file.rstrip('.wav').lstrip(model_name + '_')]
                
                # Create new filename based on the specified format
                new_filename = f"{model_letter}_{ref_id}_{script_num}.wav"
                old_path = os.path.join(model_set_path, file)
                new_path = os.path.join(output_base_path, new_filename)
                
                # Rename the file
                shutil.copyfile(old_path, new_path)

    # Save mapping to a text file
    with open(os.path.join(output_base_path, 'mapping.txt'), 'w', encoding='utf-8') as f:
        for new_name, old_name in model_letter_map.items():
            f.write(f"{new_name} -> {old_name}\n")
        for ref_id, ref_name in ref_mapping.items():
            f.write(f"{ref_name} -> {ref_id}\n")
        for script_name, script_num in script_mapping.items():
            f.write(f"{script_name} -> {script_num}\n")

    # For using in blind test web platform
    test_data = {
        test: 
            [
                {
                    "model": f"Model {model_letter.upper()}",
                    "modelId": model_name,
                    "speakers": [
                        {
                            "speaker": f"Speaker {ref_id}",
                            "scripts": [
                                {"script": f"Script {script_num}", "wav": new_filename}
                                for script_num, new_filename in enumerate(
                                    [f"{model_letter}_{ref_id}_{i}.wav" for i in range(1, len(script_mapping) + 1)], start=1
                                )
                            ]
                        }
                        for ref_id in range(1, len(REFERENCE_FILENAMES) + 1)
                    ]
                }
                for model_name, model_letter in model_letter_map.items()
                ]
            }

    test_data[test] = sorted(test_data[test], key=lambda x: x['model'])
    
    os.makedirs(os.path.join(output_base_path, f'{test}'), exist_ok=True)

    with open(os.path.join(output_base_path, f'{test}', 'data.json'), 'w', encoding='utf-8') as json_file:
        json.dump(test_data, json_file, ensure_ascii=False, indent=2)
    
    with open(os.path.join(output_base_path, f'{test}', 'script.json'), 'w', encoding='utf-8') as script_file:
        script_dict = {f"Script {num}": script for script, num in script_mapping.items()}
        json.dump(script_dict, script_file, ensure_ascii=False, indent=2)
    
    with open(os.path.join(output_base_path, f'{test}', 'speaker.json'), 'w', encoding='utf-8') as speaker_file:
        speaker_dict = {f"Speaker {num}": ref for num, ref in ref_mapping.items()}
        json.dump(speaker_dict, speaker_file, ensure_ascii=False, indent=2)

    print("Renaming complete! Mapping saved.")

Renaming complete! Mapping saved.
